### Initialize Truveta SDK
Notebook analysis for drug metrics

In [1]:
from truveta.study import Client, OutputMode, display_df
import pyspark.pandas as ps
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Use only one statement below and comment out whichever you are not using.
client = Client(output_mode = OutputMode.PandasOnSpark)
#client = Client(output_mode = OutputMode.PySpark)

In [3]:
study = client.get_study()
# Use only one statement below and comment out whichever you are not using.
population = study.get_population(title = "Delivery")
#population = study.get_population(id = "p-y4vjjpkak2tebbzl646avkf5he")

In [4]:
# Get latest completed active snapshot.
snapshot = population.get_latest_snapshot()

In [5]:
# Show tables in the snapshot.
# snapshot.get_tables()

### Medication Discontinuous 

In [6]:
# read in two files 
output_path_local = study.get_output_path(fs = True)
# read in medication with no t2d
file_to_read = output_path_local + "/medication_full.csv"
medication_full = pd.read_csv(file_to_read)
#medication_full.head()
file_to_read = output_path_local + "/test_t3.csv"
delivery_df = pd.read_csv(file_to_read)

In [7]:
delivery_df = delivery_df[(delivery_df.t2d_before_pregnancy == False)]

In [8]:
medication_df = medication_full.copy()
# medication_full.Code.value_counts()
medication_df = medication_df[medication_df['PersonId'].isin(delivery_df['PersonId'])]
medication_df = pd.merge(delivery_df[['PersonId', 'delivery_date', 'estimated_LMP']], medication_df, on = 'PersonId', how = 'left')

In [12]:
medication_df = medication_df.rename(columns={'DispenseDateTime': 'med_date'})
medication_df['med_date'] = pd.to_datetime(medication_df['med_date'])
medication_df = medication_df[medication_df['med_date'] <= medication_df['delivery_date']]
medication_df = medication_df.sort_values(['PersonId', 'med_date'])

def extract_formulation(code):
    match = re.search(r'\[(.*?)\]', code)
    if match:
        return match.group(1).strip()
    elif "Rybelsus" in code:
        return "Rybelsus"
    elif "Ozempic" in code:
        return "Ozempic"
    elif "Wegovy" in code:
        return "Wegovy"
    else:
        return "Unknown"

# Apply to your Code column
medication_df['Formulation'] = medication_df['Code'].apply(extract_formulation)

In [13]:
# check the date for medication
medication_df['med_date'] = pd.to_datetime(medication_df['med_date'])
medication_df = medication_df.groupby(['PersonId', 'med_date'], as_index=False).agg({
    'DaysSupply': 'sum',
    'Code': lambda x: ', '.join(sorted(set(x))), 
    'delivery_date': 'first', 
    'estimated_LMP' : 'first',                 
    'Formulation': lambda x: ', '.join(sorted(set(x)))
})
medication_df['EndDate'] = medication_df['med_date'] + pd.to_timedelta(medication_df['DaysSupply'], unit='d')
medication_df.head()

In [14]:
# getting EpisodeID record
records_with_episode = []

# loop through 
for pid, group in medication_df.sort_values(['PersonId', 'med_date']).groupby('PersonId'):
    group = group.reset_index(drop=True)
    episode_id = 1
    prev_end = group.loc[0, 'EndDate']
    
    group.loc[0, 'EpisodeID'] = episode_id
    records_with_episode.append(group.loc[0])
    
    for i in range(1, len(group)):
        current_start = group.loc[i, 'med_date']
        gap = (current_start - prev_end).days
        
        if gap > 60:
            episode_id += 1  # new episode
        
        group.loc[i, 'EpisodeID'] = episode_id
        prev_end = group.loc[i, 'EndDate']
        records_with_episode.append(group.loc[i])

# getting df
episode_annotated_df = pd.DataFrame(records_with_episode)

episode_annotated_df.head()

In [15]:
episode_df = episode_annotated_df.copy()

In [16]:
# sort PersonId and med date
episode_df = episode_df.sort_values(['PersonId', 'med_date'])

# getting DiscontinuationTime (the start of the next episode - previous enddate)
episode_df['NextStart'] = episode_df.groupby('PersonId')['med_date'].shift(-1)
episode_df['DiscontinuationTime'] = (episode_df['NextStart'] - episode_df['EndDate']).dt.days

# check if there is reinitiation: current gap > 60 and there is also the next startdate then == reinitiation
episode_df['IsReinitiation'] = (episode_df['DiscontinuationTime'] > 60)  & (episode_df['NextStart'].notna())

episode_df.head()

In [17]:
summary = episode_df.groupby('PersonId').agg(
    DrugStart=('med_date', 'min'),
    DrugEnd=('EndDate', 'max'),
    NumDrugEpisodes=('EpisodeID', lambda x: len(set(x))),
    Discontinued60=('DiscontinuationTime', lambda x: any(x > 60)),
    TotalDiscontinuationTime=('DiscontinuationTime', lambda x: x[x > 60].sum()), # getting how long the drug is discont.
    Reinitiation=('IsReinitiation', 'any')
).reset_index()


In [18]:
summary

In [19]:
delivery_df[delivery_df.PersonId == '041f59e2-3836-07bd-f7e3-86833f6e7601']

In [20]:
episode_df[episode_df.PersonId == '041f59e2-3836-07bd-f7e3-86833f6e7601']

#### Drug Formulation change - not require Don't Run!

In [29]:
# sort
episode_df = episode_df.sort_values(['PersonId', 'med_date'])

# getting previous formulation
episode_df['PrevFormulation'] = episode_df.groupby('PersonId')['Formulation'].shift(1)

# compare drug change
def detect_change(row):
    if pd.isna(row['PrevFormulation']):
        return None
    if row['Formulation'] != row['PrevFormulation']:
        return f"{row['PrevFormulation']} -> {row['Formulation']}"
    else:
        return None

episode_df['DrugChange'] = episode_df.apply(detect_change, axis=1)


In [30]:
episode_df.head()

In [31]:
episode_df.DrugChange.value_counts()

In [32]:
len(summary)

### Getting Accumulated persistence time before delivery

In [21]:
delivery_episode_summary = episode_df.groupby(['PersonId', 'EpisodeID']).agg(
    EpisodeStart=('med_date', 'min'),
    EpisodeEnd=('EndDate', 'max'),
    delivery_date=('delivery_date', 'first')
).reset_index()

delivery_episode_summary.head(10)

In [22]:
# cut off EpisodeEnd before delivery, which it should already did 
delivery_episode_summary['delivery_date'] = pd.to_datetime(delivery_episode_summary['delivery_date'])
delivery_episode_summary['EpisodeEnd_Truncated'] = delivery_episode_summary[['EpisodeEnd', 'delivery_date']].min(axis=1)

# only keep episode before delivery
delivery_episode_summary = delivery_episode_summary[delivery_episode_summary['EpisodeStart'] <= delivery_episode_summary['delivery_date']]
delivery_episode_summary.head(10)

In [23]:
# compute the duration for each episode
delivery_episode_summary['EpisodeDurationBeforeDelivery'] = (delivery_episode_summary['EpisodeEnd_Truncated'] - delivery_episode_summary['EpisodeStart']).dt.days

# getting all the persistence time before delivery (sum up)
accumulated_delivery = delivery_episode_summary.groupby('PersonId').agg(
    AccumulatedPersistenceBeforeDelivery=('EpisodeDurationBeforeDelivery', 'sum')
).reset_index()

In [24]:
delivery_episode_summary.head(10)

In [25]:
accumulated_delivery.head(10)

### Accumulated persistence time before pregnancy

In [26]:
preg_episode_summary = episode_df.groupby(['PersonId', 'EpisodeID']).agg(
    EpisodeStart=('med_date', 'min'),
    EpisodeEnd=('EndDate', 'max'),
    estimated_LMP=('estimated_LMP', 'first')
).reset_index()

# Stop episode @ estimated_LMP
preg_episode_summary['estimated_LMP'] = pd.to_datetime(preg_episode_summary['estimated_LMP'])
preg_episode_summary['EpisodeEnd_Truncated'] = preg_episode_summary.apply(
    lambda row: min(row['EpisodeEnd'], row['estimated_LMP']) if pd.notna(row['EpisodeEnd']) and pd.notna(row['estimated_LMP']) else pd.NaT,
    axis=1
)

# keep episode before
preg_episode_summary = preg_episode_summary[preg_episode_summary['EpisodeStart'] <= preg_episode_summary['estimated_LMP']]

# compute the duration
preg_episode_summary['EpisodeDurationBeforePregnancy'] = (
    preg_episode_summary['EpisodeEnd_Truncated'] - preg_episode_summary['EpisodeStart']
).dt.days

# getting the persistence
accumulated_preg_persistence = preg_episode_summary.groupby('PersonId').agg(
    AccumulatedPersistenceBeforePregnancy=('EpisodeDurationBeforePregnancy', 'sum')
).reset_index()

In [27]:
len(preg_episode_summary), len(preg_episode_summary.PersonId.unique())

In [28]:
preg_episode_summary.head(10)

In [29]:
accumulated_preg_persistence.head(10)

### Drug Exposure

In [30]:
check_preg = episode_annotated_df.copy()

check_preg['estimated_LMP'] = pd.to_datetime(check_preg['estimated_LMP'])
check_preg['EndDate'] = pd.to_datetime(check_preg['EndDate'])
check_preg['med_date'] = pd.to_datetime(check_preg['med_date'])

def pregnancy_exposure(row):
    start = row['med_date']
    end = row['EndDate']
    preg_start = row['estimated_LMP']
    
    if end <= preg_start:
        return (False, 0)
    
    if start >= preg_start:
        exposure_days = (end - start).days
    else:
        exposure_days = (end - preg_start).days

    return (True, exposure_days)

check_preg[['WasExposedInPregnancy', 'PregnancyExposureDays']] = check_preg.apply(
    pregnancy_exposure, axis=1, result_type='expand'
)

check_preg.head()

In [31]:
check_preg.head(20)

In [32]:
# getting all the exposure date and sum them up
pregnancy_df = check_preg.groupby('PersonId').agg(
    ExposedInPregnancy=('WasExposedInPregnancy', 'any'),
    TotalExposureDaysInPregnancy=('PregnancyExposureDays', 'sum')
).reset_index()


In [33]:
pregnancy_df

In [34]:
summary_df = pd.merge(summary, accumulated_delivery, on = 'PersonId')
summary_df = pd.merge(summary_df, accumulated_preg_persistence, on = 'PersonId')
summary_df = pd.merge(summary_df, pregnancy_df, on = 'PersonId')

summary_df.head()

In [35]:
summary_df[summary_df.PersonId == '2055914d-c62e-bd79-7bac-342de880bbab']
len(summary), len(accumulated_preg_persistence), len(pregnancy_df)

In [36]:
len(summary_df[summary_df['AccumulatedPersistenceBeforePregnancy'] == 0])

In [37]:
summary_df['AccumulatedPersistenceBeforePregnancy'] = summary_df['AccumulatedPersistenceBeforePregnancy'].fillna(0)

In [38]:
from statsmodels.formula.api import ols
! pip install tableone
from tableone import TableOne
import statsmodels.api as sm

In [39]:
summary_df.columns

In [40]:
table1 = summary_df.copy()
t1 = table1[['PersonId', 'NumDrugEpisodes', 'Discontinued60',
       'TotalDiscontinuationTime', 'Reinitiation',
       'AccumulatedPersistenceBeforeDelivery',
       'AccumulatedPersistenceBeforePregnancy', 'ExposedInPregnancy',
       'TotalExposureDaysInPregnancy']]
columns_df = ['NumDrugEpisodes', 'Discontinued60',
       'TotalDiscontinuationTime', 'Reinitiation',
       'AccumulatedPersistenceBeforeDelivery',
       'AccumulatedPersistenceBeforePregnancy', 'ExposedInPregnancy',
       'TotalExposureDaysInPregnancy']

cate_df = ['Discontinued60', 'Reinitiation','ExposedInPregnancy']

table1 = TableOne(t1, columns=columns_df, categorical=cate_df, pval=False)
table1

In [41]:
table1 = summary_df.copy()
t1 = table1[['PersonId', 'NumDrugEpisodes', 'Discontinued60',
       'TotalDiscontinuationTime', 'Reinitiation',
       'AccumulatedPersistenceBeforeDelivery',
       'AccumulatedPersistenceBeforePregnancy', 'ExposedInPregnancy',
       'TotalExposureDaysInPregnancy']]
columns_df = ['NumDrugEpisodes', 
       'TotalDiscontinuationTime', 'Reinitiation',
       'AccumulatedPersistenceBeforeDelivery',
       'AccumulatedPersistenceBeforePregnancy', 'ExposedInPregnancy',
       'TotalExposureDaysInPregnancy']

cate_df = ['Reinitiation','ExposedInPregnancy']

table1 = TableOne(t1, columns=columns_df, categorical=cate_df, groupby = 'Discontinued60', pval=True)
table1

In [42]:
output_path_local = study.get_output_path(fs = True)
file_to_write = output_path_local + "/revise_results/drugexposurenot2d.csv"
table1.to_csv(file_to_write, index = True)

In [44]:
cols = [
    "AccumulatedPersistenceBeforePregnancy",
    "TotalExposureDaysInPregnancy"
]

summary = summary_df[cols].agg(['median', 
                          lambda x: x.quantile(0.25), 
                          lambda x: x.quantile(0.75)])

summary.index = ['Median', 'Q1', 'Q3']
summary